# pandas 02. 選ぶ・絞る・変える

`[]` と `.loc` の違い、代入でハマるところ、値を変える書き方の使い分けを見ます。

進み方は `pandas-01` と同じです。
**セルA と セルB を見比べる → 答え合わせ → ミニ練習を1問**。

最初のセルで、`pandas-01` でやった読み込みをまとめて済ませておきます。
中身はもう読めるはずです。

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

NULLISH = {"NULL", "N/A", "-", ""}

# orders.csv を、型を整えた状態で読む (pandas-01 でやったこと)
def load():
    df = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
    df = df.map(lambda s: None if str(s).strip() in NULLISH else s)
    df["qty"] = pd.to_numeric(df["qty"]).astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False)).astype("Int64")
    return df

df = load()
df

---
## 1. 列を選ぶ

### 1-1. df['col'] と df[['col']]

角括弧が1つか2つかで、返ってくるものが変わります。

In [ ]:
# A: 1つ
a = df["amount"]
print(type(a).__name__)
a.head(3)

In [ ]:
# B: 2つ
b = df[["amount"]]
print(type(b).__name__)
b.head(3)

<details>
<summary>答え合わせ</summary>

- `A` は **Series**(1列)。`.sum()` や `.str` が使えます
- `B` は **DataFrame**(表)。列が1つだけの表です

リストを渡せば複数列を選べて、**その順に並びます**。

```python
df[["amount", "order_id"]]      # この順になる
```

計算に使うなら `A`、表として扱うなら `B`、という使い分けです。

</details>

In [ ]:
# ミニ練習: order_id と amount の2列を、この順で取り出す

ans = ...   # ここに書く

assert list(ans.columns) == ["order_id", "amount"], list(ans.columns)
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[["order_id", "amount"]]
```

</details>

### 1-2. .loc と .iloc

行や列を「名前で指す」か「位置で指す」かの違いです。

In [ ]:
# A: .loc (ラベル)
print(df.loc[0:2, "order_id"])
print("---")
print(df.loc[df["region"] == "east", ["order_id", "amount"]])

In [ ]:
# B: .iloc (位置)
print(df.iloc[0:2, 0])
print("---")
print(df.iloc[[0, 2], [0, 6]])

<details>
<summary>答え合わせ</summary>

**`.loc` のスライスは右端を含みます。** `0:2` は3行(0,1,2)。
`.iloc` は Python のふつうのスライスなので `0:2` は2行(0,1)です。

| | 指定するもの | スライスの右端 |
| --- | --- | --- |
| `.loc` | ラベル(index の値、列名) | **含む** |
| `.iloc` | 位置(0始まりの整数) | 含まない |

いまは index が 0,1,2... なので `.loc` でも整数が使えますが、
並べ替えたり index を振り直したりすると一致しなくなります。

実際いちばんよく使うのは `df.loc[条件, 列]` の形です。

</details>

In [ ]:
# ミニ練習: .loc で、region が "north" の行の order_id を取り出す

ans = ...   # ここに書く

assert ans.tolist() == ["O-005", "O-006"], ans.tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.loc[df["region"] == "north", "order_id"]
```

</details>

---
## 2. 行を絞る

### 2-1. and は使えない

複数条件をつなぐとき、Python の `and` ではなく `&` を使います。

In [ ]:
# A: and (落ちる)
try:
    df[(df["region"] == "east") and (df["status"] == "completed")]
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# B: & と括弧
df[(df["region"] == "east") & (df["status"] == "completed")]

<details>
<summary>答え合わせ</summary>

`and` は左右を**1個の真偽値**に変えようとします。Series は
「全部Trueか」「1つでもTrueか」が決まらないので `ValueError` になります。

`&` `|` `~` は**要素ごと**に働くので、こちらを使います。

括弧が要るのは演算子の優先順位のためです。`&` は `==` より強いので、
括弧を外すと式の意味が変わってしまいます。

**条件はいつも括弧で囲む**、とだけ覚えておけば大丈夫です。

</details>

In [ ]:
# ミニ練習: status が "completed" で、かつ qty が 2 以上の行を取り出す

ans = ...   # ここに書く

assert len(ans) == 5, f"5行のはず: {len(ans)}"
assert ans["order_id"].tolist() == ["O-001", "O-003", "O-006", "O-007", "O-009"]
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[(df["status"] == "completed") & (df["qty"] >= 2)]
```

</details>

### 2-2. isin / between / query

同じ絞り込みでも、読みやすい書き方があります。

In [ ]:
# A: & や | でつなぐ
a = df[(df["region"] == "east") | (df["region"] == "west")]
display(a)
b = df[(df["amount"] >= 1000) & (df["amount"] <= 3000)]
b

In [ ]:
# B: isin / between / query
a = df[df["region"].isin(["east", "west"])]
display(a)
b = df[df["amount"].between(1000, 3000)]
display(b)
c = df.query("region in ['east','west'] and amount >= 1000")
c

<details>
<summary>答え合わせ</summary>

結果は同じです。条件が増えるほど `B` のほうが読みやすくなります。

- `between(a, b)` は**両端を含みます**(`inclusive="both"` が既定)。
  片側だけにしたいなら `inclusive="left"` など
- `query` は文字列で書きます。中では `and` / `or` が使えます。
  外の変数を参照するときは `@` を付けます: `df.query("amount > @limit")`

短い条件は `query`、複雑なものは `&`、くらいの気分で選んで問題ありません。

</details>

In [ ]:
# ミニ練習: region が "east" か "north" の行を、isin を使って取り出す

ans = ...   # ここに書く

assert len(ans) == 6, f"6行のはず: {len(ans)}"
assert set(ans["region"]) == {"east", "north"}
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[df["region"].isin(["east", "north"])]
```

</details>

### 2-3. 欠損を含む列で比較すると

条件式が欠損をどう扱うか、という話です。

In [ ]:
# A: 比較する
print(df["amount"].isna().sum(), "件が欠損")
a = df[df["amount"] > 1000]
print(len(a), "行")
a[["order_id", "amount"]]

In [ ]:
# B: 否定してみる
b = df[~(df["amount"] > 1000)]
print(len(b), "行")
b[["order_id", "amount"]]

<details>
<summary>答え合わせ</summary>

**欠損の行は A にも B にも入りません。** 足しても全行になりません。

欠損との比較は「True でも False でもない」ので、`~` で反転しても拾われないためです。

欠損も残したいときは、その分を明示的に足します。

```python
df[(df["amount"] > 1000) | df["amount"].isna()]
```

「条件の否定」が「残り全部」にならない、というのは行数が合わない事故の常連です。

</details>

In [ ]:
# ミニ練習: amount が 1000 以下、もしくは欠損の行を取り出す

ans = ...   # ここに書く

assert len(ans) == 5, f"5行のはず: {len(ans)}"
assert "O-006" in ans["order_id"].tolist(), "欠損の行が入っていない"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[(df["amount"] <= 1000) | df["amount"].isna()]
```

</details>

---
## 3. 代入でハマるところ

### 3-1. 絞ってから代入する

「絞った結果に代入する」を2通りで書いてみます。片方は効かないことがあります。

In [ ]:
# A: 絞ってから代入
a = load()
sub = a[a["region"] == "east"]
sub["region"] = "EAST"       # 警告が出るかもしれない
print("元の a は変わったか:")
print(a[a["region"].isin(["east", "EAST"])]["region"].tolist())

In [ ]:
# B: .loc で一度に指定
b = load()
b.loc[b["region"] == "east", "region"] = "EAST"
print("元の b は変わったか:")
print(b["region"].tolist())

<details>
<summary>答え合わせ</summary>

`A` の `sub` は、元の `a` の**コピーかもしれないし、参照かもしれません**。
どちらになるかは pandas の内部事情で決まるので、元が変わるかどうかが読めません。
pandas はこれを `SettingWithCopyWarning` で知らせてくれます。

`B` は「どの行の、どの列に」を1回の `.loc` で指定しているので、迷いがありません。

覚えることは2つだけです。

- **元を変えたい** → `df.loc[条件, 列] = 値`
- **別物を作りたい** → `sub = df[条件].copy()` と `.copy()` を付ける

</details>

In [ ]:
# ミニ練習: region が "WEST" の行を "west" に直す (.loc を使う)

c = load()

# ここに書く

assert (c["region"] == "WEST").sum() == 0
assert (c["region"] == "west").sum() == 4
print("OK")

<details>
<summary>答え</summary>

```python
c.loc[c["region"] == "WEST", "region"] = "west"
```

</details>

### 3-2. assign は元を変えない

列を足す書き方が2つあります。

In [ ]:
# A: 直接代入
a = load()
a["total"] = a["qty"] * a["amount"]
print(a.columns.tolist())

In [ ]:
# B: assign
b = load()
c = b.assign(total=b["qty"] * b["amount"])
print("元:", b.columns.tolist())
print("新:", c.columns.tolist())

<details>
<summary>答え合わせ</summary>

`A` は元の `a` を書き換えます。`B` の `assign` は**新しい表を返します**。

`assign` の良いところは、メソッドチェーンの途中に置けることです。

```python
(df
 .assign(total=lambda d: d["qty"] * d["amount"])
 .query("total > 3000")
 .sort_values("total"))
```

`lambda d:` を使うと、**その時点の DataFrame** を参照できます。

短いコードなら `A` で十分です。変換が何段も続くときに `B` を思い出してください。

</details>

In [ ]:
# ミニ練習: assign で unit_price (= amount / qty) を足した新しい表を作る
#           (元の df は変えない)

ans = ...   # ここに書く

assert "unit_price" not in df.columns, "元の df を変えている"
assert "unit_price" in ans.columns
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.assign(unit_price=df["amount"] / df["qty"])
```

</details>

---
## 4. 値を変える

### 4-1. map / apply / .str

1列の全部の値に処理をかける書き方です。欠損の扱いが違います。

In [ ]:
# A: map と apply
s = df["region"]
print("map      :", s.map(lambda x: x.upper() if x else x).tolist())
print("apply    :", s.apply(lambda x: x.upper() if x else x).tolist())
print("---- 欠損を含む列に、そのままかけてみる ----")
t = df["customer_id"]
try:
    print(t.map(str.upper).tolist())
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# B: .str アクセサ
t = df["customer_id"]
print(t.str.upper().tolist())
print("---")
print(df["region"].str.lower().tolist())

<details>
<summary>答え合わせ</summary>

- `Series.map` と `Series.apply` は、1列に対してはほぼ同じものです。
  `map` は辞書も受け取れます(`s.map({"east": "東"})`)
- **`.str.*` は欠損を素通りさせます。** `None` はそのまま `None` で返ります

`map(str.upper)` は欠損に対しても `str.upper(None)` を呼ぼうとして落ちます。
`map` に `na_action="ignore"` を付けると飛ばしてくれます。

文字列の操作は `.str` を最初の選択肢にしておくと、欠損で悩む回数が減ります。

</details>

In [ ]:
# ミニ練習: status を大文字にした Series を作る (.str を使う)

ans = ...   # ここに書く

assert ans.tolist()[:3] == ["COMPLETED", "CANCELLED", "COMPLETED"], ans.tolist()[:3]
print("OK")

<details>
<summary>答え</summary>

```python
ans = df["status"].str.upper()
```

</details>

### 4-2. apply の axis

DataFrame に `apply` をかけると、列ごとに動くか行ごとに動くかを選べます。

In [ ]:
# A: axis=0 (既定) … 列ごと
a = df[["qty", "amount"]].apply(lambda col: col.max())
print(type(a).__name__)
a

In [ ]:
# B: axis=1 … 行ごと
b = df[["qty", "amount"]].apply(lambda row: row["qty"] * row["amount"], axis=1)
print(type(b).__name__)
b.head()

<details>
<summary>答え合わせ</summary>

- `axis=0`(既定)は**列ごと**に関数を呼びます。渡ってくるのは1列ぶんの Series
- `axis=1` は**行ごと**。渡ってくるのは1行ぶんの Series

覚えにくいので、迷ったら小さいデータで両方試すのがいちばん早いです。

なお `axis=1` の `apply` は1行ずつ Python の関数を呼ぶので**かなり遅い**です。
`df["qty"] * df["amount"]` と書けるならそちらを使います。

</details>

In [ ]:
# ミニ練習: qty と amount の「列ごとの最小値」を出す

ans = ...   # ここに書く

assert ans["qty"] == 0, ans.to_dict()
assert ans["amount"] == 0
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[["qty", "amount"]].apply(lambda col: col.min())
```

</details>

### 4-3. where / mask / np.where

条件で値を差し替える3つです。「どちらを残すか」が逆なので、そこだけ注意します。

In [ ]:
# A: where と mask
s = df["amount"]
print("where:", s.where(s > 1000, 0).tolist())
print("mask :", s.mask(s > 1000, 0).tolist())

In [ ]:
# B: np.where
s = df["amount"]
try:
    print(np.where(s > 1000, "高", "低"))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# 欠損を先に埋めれば通る
print(np.where(s.fillna(0) > 1000, "高", "低"))

<details>
<summary>答え合わせ</summary>

- `s.where(cond, other)` は **cond が True の値を残します**(False を `other` に)
- `s.mask(cond, other)` はその逆で、**True の値を差し替えます**
- `np.where(cond, a, b)` は三項演算子。True なら `a`、False なら `b`

`where` の引数は直感に反しやすいので、**「where は残す、mask は隠す」**と覚えます。

`np.where` は `Int64` のような欠損を持てる型が苦手です(`pd.NA` の真偽を決められません)。
`fillna` で埋めるか、pandas 側の `where` / `mask` を使ってください。

</details>

In [ ]:
# ミニ練習: amount が 3000 より大きい値を 3000 に丸めた Series を作る
#           (mask を使う)

ans = ...   # ここに書く

assert ans.max() == 3000, ans.tolist()
assert ans.tolist()[0] == 2400
print("OK")

<details>
<summary>答え</summary>

```python
ans = df["amount"].mask(df["amount"] > 3000, 3000)
```

</details>

### 4-4. replace と str.replace

名前は似ていますが、別物です。

In [ ]:
# A: replace (値まるごと)
s = pd.Series(["east", "west", "eastern"])
print(s.replace("east", "E").tolist())
print(s.replace({"east": "E", "west": "W"}).tolist())

In [ ]:
# B: str.replace (部分文字列)
s = pd.Series(["east", "west", "eastern"])
print(s.str.replace("east", "E", regex=False).tolist())
print(s.str.replace(r"^e", "E", regex=True).tolist())

<details>
<summary>答え合わせ</summary>

- `Series.replace` は**値そのもの**が一致したときだけ置き換えます。
  `"eastern"` は変わりません。辞書で対応表を渡せます
- `Series.str.replace` は**部分文字列**を置き換えます。`"eastern"` → `"Eern"` になります

`str.replace` の `regex` は pandas 2.x では既定が `False` です。
正規表現を使いたいときは `regex=True` を明示します。

コード体系の言い換え(`E` → `east`)は `replace` の辞書、
表記の掃除(カンマや円記号を落とす)は `str.replace`、という住み分けです。

</details>

In [ ]:
# ミニ練習: replace の辞書で "gold" を "G"、"silver" を "S" に置き換える

s = pd.Series(["gold", "silver", "bronze"])

ans = ...   # ここに書く

assert ans.tolist() == ["G", "S", "bronze"], ans.tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = s.replace({"gold": "G", "silver": "S"})
```

</details>

---
## 仕上げ

In [ ]:
# 仕上げ1: status が completed で、amount が 1000 以上の行を、
#          order_id と amount の2列だけにして取り出す。

ans = ...   # ここに書く

assert list(ans.columns) == ["order_id", "amount"], f"列が違う: {list(ans.columns)}"
assert len(ans) == 5, f"5行のはず: {len(ans)}"
assert ans["amount"].sum() == 18000
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.loc[
    (df["status"] == "completed") & (df["amount"] >= 1000),
    ["order_id", "amount"],
]
```

</details>

In [ ]:
# 仕上げ2: region を小文字にそろえた列 region_norm を、
#          元の df を書き換えずに追加した新しい表を作る。
#          (欠損があっても落ちないこと)

ans = ...   # ここに書く

assert "region_norm" not in df.columns, "元の df を書き換えている"
assert sorted(ans["region_norm"].unique()) == ["east", "north", "south", "west"]
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.assign(region_norm=df["region"].str.lower())
```

</details>

In [ ]:
# 仕上げ3: amount が 2000 以上なら "high"、2000 未満なら "low"、
#          欠損なら "unknown" になる Series を作る。

ans = ...   # ここに書く

assert ans.tolist() == ["high", "low", "high", "high", "low", "unknown",
                        "high", "low", "high", "low", "low", "low"], ans.tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = (
    pd.Series("low", index=df.index)
    .mask(df["amount"] >= 2000, "high")
    .mask(df["amount"].isna(), "unknown")
)
```

</details>

---
## まとめ

| 書き方 | 意味 | 注意 |
| --- | --- | --- |
| `df["c"]` | Series | 1列 |
| `df[["c"]]` | DataFrame | リストの順に並ぶ |
| `.loc[行, 列]` | ラベルで指定 | **スライスの右端を含む** |
| `.iloc[行, 列]` | 位置で指定 | 右端を含まない |
| `&` `\|` `~` | 要素ごとの論理演算 | **括弧が要る**。`and` は使えない |
| `isin` / `between` | 読みやすい絞り込み | `between` は両端を含む |
| `df.loc[cond, col] = v` | 安全な代入 | 角括弧を2回続けない |
| `.copy()` | 明示的にコピー | 別物として扱いたいとき |
| `assign` | 列を足した新しい表を返す | チェーンの途中に置ける |
| `.str.*` | 文字列操作 | **欠損を素通りする**。第一候補 |
| `map(na_action="ignore")` | 欠損を飛ばす | `.str` で足りないとき |
| `apply(axis=1)` | 行ごと | 遅い。まず列の計算で書けないか考える |
| `where` / `mask` | 残す / 隠す | 意味が逆 |
| `replace` / `str.replace` | 値ごと / 部分文字列 | `regex=` を明示する |

次は `pandas-03-group-and-dedup.ipynb` です。